md

In [ ]:
# Registered constants, read from the pre-registration at build time.
SEEDS_HERE = [1401, 1402]
STOP_SPREAD_HERE = 0.15
print('registered seeds:', SEEDS_HERE, '| stopping threshold:', STOP_SPREAD_HERE)

In [ ]:
import glob, json, os, shutil, subprocess, sys

work = '/kaggle/working/kalia'
if not os.path.exists(work):
    hits = sorted(glob.glob('/kaggle/input/**/train.py', recursive=True))
    assert hits, 'attach the kalia-code-dev dataset'
    shutil.copytree(os.path.dirname(hits[0]), work)
os.chdir(work)
data = sorted(glob.glob('/kaggle/input/**/train.bin', recursive=True))
assert data, 'train.bin not found - attach a prep output as a kernel source'
DATA = os.path.dirname(data[0])
print('code:', work)
print('data:', DATA)
import torch
print('gpu:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('adaptive_every supported:', 'adaptive_every' in open('train.py').read())

In [ ]:
arms = ['v030-replay-00-s1401', 'v030-replay-10-s1401', 'v030-replay-40-s1401']
for a in arms:
    assert os.path.exists(f'configs/{a}.yaml'), f'missing configs/{a}.yaml'
    print(a)
print('\nRun Step 2 first: the LR schedule must be fixed before a ratio is')
print('meaningful, and Step 1 supplies the bar.')

In [ ]:
print(subprocess.run([sys.executable, 'ablate.py', '--arms', ','.join(arms),
                       '--steps', '500', '--data-dir', DATA,
                       '--out-root', '/kaggle/working/out'],
                      capture_output=True, text=True).stdout[-6000:])

In [ ]:
import pandas as pd
BAR = json.load(open('/kaggle/working/step1_spread.json'))['bar']
losses = {}
for a in arms:
    df = pd.read_csv(f'/kaggle/working/out/{a}/val_log.csv')
    print(f'{a:26s} ' + '  '.join(f"{int(r.step)}={r.val_loss:.4f}" for r in df.itertuples()))
    losses[a] = float(df.iloc[-1].val_loss)
base = losses['v030-replay-00-s1401']
print(f'\nbaseline 0%: {base:.4f} | bar {BAR:.4f}')
best_arm, best = None, None
for a in arms[1:]:
    d = losses[a] - base
    mark = 'BEATS' if d <= -BAR else 'within noise'
    print(f'  {a:26s} {losses[a]:.4f}  delta {d:+.4f}  {mark}')
    if best is None or losses[a] < best:
        best, best_arm = losses[a], a
matters = best is not None and (base - best) > BAR
print(f'\nratio matters here: {"YES" if matters else "NO"} '
      f'(best {best_arm})')
if not matters:
    print('  All three within the measured spread. That is a result: at 58M the')
    print('  replay ratio is not the delicate knob the folklore suggests, which')
    print('  is what the 405M paper reports too -- at a scale nobody extrapolated to.')